# CS 198/199 
# Joaquin B. Salvador
# Explainable AI for Post-COVID Psychological Profile Prediction

## 2.0
### Keras

https://github.com/shap/shap/blob/master/notebooks/tabular_examples/neural_networks/Census%20income%20classification%20with%20Keras.ipynb

In [ ]:
# -------------------------------------
# Library configuration
# -------------------------------------
# Standard libraries
import numpy as np
import pandas as pd

In [ ]:
# Loading dataset
df = pd.read_csv('../data/processed/data.csv')

In [ ]:
# Displaying dataset
df

In [ ]:
# Sorting dataset based on SAMPLEID
df_sorted = df.sort_values(by=['SAMPLEID'])

In [ ]:
# Displaying sorted dataset
df_sorted

In [ ]:
# Dropping empty rows from sorted dataset
df_cleaned = df_sorted.dropna()

In [ ]:
# Displaying cleaned dataset
df_cleaned

In [ ]:
# Gathering column names
column_names = df_cleaned.columns.tolist()

In [ ]:
# Displaying column names
column_names

In [ ]:
# Setting targets
#df_targets = df_cleaned[['6_K6_total',
# '6_PHQ9_total',
# '6_GAD7_total',
# '6_SSS8_total',
# '6_PTGI-X(Q7_22_27)',
# '6_SHS_total',
# '6_UCLA_total',
# '6_LSNS6_total',
# '6_AUDIT']]

df_target = df_cleaned[['6_K6_total']]

#num_targets = len(df_targets.columns.tolist())

In [ ]:
# Setting features
df_features = df_cleaned.drop([
 'SAMPLEID',
 'ANSWERDATE',
'5_PREFECTURE',
 '6_K6_total',
 '6_PHQ9_total',
 '6_GAD7_total',
 '6_SSS8_total',
 '6_PTGI-X(Q7_22_27)',
 '6_SHS_total',
 '6_UCLA_total',
 '6_LSNS6_total',
 '6_AUDIT'], axis=1)

num_features = len(df_features.columns.tolist())

# KERAS IMPLEMENTATION

In [2]:
from keras.layers import (
    Dense,
    Dropout,
    Flatten,
    Input,
    concatenate,
)
from keras.layers.embeddings import Embedding
from keras.models import Model
from sklearn.model_selection import train_test_split

import shap

# print the JS visualization code to the notebook
shap.initjs()

ImportError: /home/joaquin/anaconda3/lib/python3.11/site-packages/tensorflow/python/platform/../../libtensorflow_framework.so.2: undefined symbol: _ZTIN6snappy4SinkE

In [ ]:
X = df_features
y = df_target

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=7)

In [ ]:
# build model
input_els = []
encoded_els = []
for k, dtype in dtypes:
    input_els.append(Input(shape=(1,)))
    if dtype == "int8":
        e = Flatten()(Embedding(X_train[k].max() + 1, 1)(input_els[-1]))
    else:
        e = input_els[-1]
    encoded_els.append(e)
encoded_els = concatenate(encoded_els)
layer1 = Dropout(0.5)(Dense(100, activation="relu")(encoded_els))
out = Dense(1)(layer1)

# train model
regression = Model(inputs=input_els, outputs=[out])
regression.compile(optimizer="adam", loss="binary_crossentropy")
regression.fit(
    [X_train[k].values for k, t in dtypes],
    y_train,
    epochs=50,
    batch_size=512,
    shuffle=True,
    validation_data=([X_valid[k].values for k, t in dtypes], y_valid),
)


In [ ]:
def f(X):
    return regression.predict([X[:, i] for i in range(X.shape[1])]).flatten()

In [ ]:
explainer = shap.KernelExplainer(f, X.iloc[:50, :])
shap_values = explainer.shap_values(X.iloc[299, :], nsamples=500)
shap.force_plot(explainer.expected_value, shap_values, X_display.iloc[299, :])

In [ ]:
shap_values50 = explainer.shap_values(X.iloc[280:330, :], nsamples=500)

In [ ]:
shap.force_plot(explainer.expected_value, shap_values50, X_display.iloc[280:330, :])